<a href="https://colab.research.google.com/github/KN-Vignesh/AI-Projects/blob/main/Combined_metric_Calc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install langchain==0.3.25
!pip -q install langchain-openai==0.3.16
!pip -q install langchain-chroma==0.2.4
!pip -q install chromadb==1.0.15
!pip -q install deepeval==4.1.3
!pip -q install pandas
!pip -q install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 44.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.5.6 which is incompatible.
langchain-openai 0.3.16 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.5.6 which is incompatible.


In [ ]:
import os
import pandas as pd

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from operator import itemgetter

In [ ]:
from google.colab import userdata
key = userdata.get('ACL')

In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-2",
    google_api_key = key
)

In [ ]:
# go to https://drive.google.com/file/d/1QkSY9W5RyaBnY8c5FLIsmpPVXoHTQ-fb/view?usp=sharing download it
# manually upload it on colab
!gdown 1QkSY9W5RyaBnY8c5FLIsmpPVXoHTQ-fb


Downloading...
From: https://drive.google.com/uc?id=1QkSY9W5RyaBnY8c5FLIsmpPVXoHTQ-fb
To: /content/rag_eval_docs.csv
100% 2.66k/2.66k [00:00<00:00, 11.2MB/s]


In [ ]:
import pandas as pd
df = pd.read_csv('/content/rag_eval_docs.csv')
df


,id,title,context
0,1,Machine Learning,Machine learning is a field of artificial inte...
1,2,Deep Learning,Deep learning is a subset of machine learning ...
2,3,Natural Language Processing (NLP),NLP is a branch of AI that enables computers t...
3,4,Pyramids,"Pyramids are ancient structures, often serving..."
4,5,Photosynthesis,Photosynthesis is the process plants use to co...
5,6,Biology,"Biology is the study of living organisms, cove..."
6,7,Quantum Mechanics,Quantum mechanics is a branch of physics that ...
7,8,Cryptocurrency,Cryptocurrency is a digital currency that uses...
8,9,Renewable Energy,"Renewable energy sources, such as solar and wi..."
9,10,Artificial Intelligence,Artificial intelligence refers to machines mim...


In [ ]:
docs = df.to_dict(orient='records')
docs[:3]


[{'id': 1,
  'title': 'Machine Learning',
  'context': 'Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.'},
 {'id': 2,
  'title': 'Deep Learning',
  'context': 'Deep learning is a subset of machine learning utilizing neural networks with many layers. It excels in complex tasks like image and speech recognition. Convolutional and recurrent neural networks are among the common architectures used.'},
 {'id': 3,
  'title': 'Natural Language Processing (NLP)',
  'context': 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.'}]

In [ ]:
from langchain.docstore.document import Document
processed_docs = []


for doc in docs:
  metadata = {
      "title": doc['title'],
      "id": doc['id']
  }
  data = doc['context']
  processed_docs.append(Document(page_content=data, metadata=metadata))
processed_docs[:3]


[Document(metadata={'title': 'Machine Learning', 'id': 1}, page_content='Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.'),
 Document(metadata={'title': 'Deep Learning', 'id': 2}, page_content='Deep learning is a subset of machine learning utilizing neural networks with many layers. It excels in complex tasks like image and speech recognition. Convolutional and recurrent neural networks are among the common architectures used.'),
 Document(metadata={'title': 'Natural Language Processing (NLP)', 'id': 3}, page_content='NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.')]

In [ ]:
from langchain_chroma import Chroma
chroma_db = Chroma.from_documents(documents=processed_docs,
                                  collection_name='my_db',embedding=embeddings,
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory='./my_db')


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
# Load Vector DB from disk
from langchain_chroma import Chroma
chroma_db = Chroma(persist_directory='./my_db', collection_name='my_db',
                   embedding_function=embeddings)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
# Create Retriever
similarity_retriever = chroma_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k":3,
        "score_threshold":0.30
    }
)


In [ ]:
# Display Retrieved Documents
from IPython.display import display, Markdown

def display_docs(docs):
    for doc in docs:
        print("Metadata :", doc.metadata)
        print("Content :")
        display(Markdown(doc.page_content))
        print()


In [ ]:
import os
from deepeval.models import GeminiModel
from langchain_google_genai import ChatGoogleGenerativeAI

# Set the Google API key as an environment variable for DeepEval
os.environ['GOOGLE_API_KEY'] = key

# Instantiate DeepEval's native Gemini model wrapper
deepeval_gemini_model = GeminiModel(
    model="gemini-3.5-flash",
    api_key=key,
    temperature=0.0
)

# Instantiate a LangChain-compatible Gemini model
langchain_gemini_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash", google_api_key=key, temperature=0.0)

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
"""
You are a helpful AI assistant.

Answer ONLY from the provided context.

Context:
{context}

Question:
{question}
"""
)


In [ ]:
# Format Retrieved Documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# Re-initialize the response chain to ensure it uses the corrected langchain_gemini_model
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

src_rag_response_chain = (
    {
        "context": itemgetter("context") | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | langchain_gemini_model
    | StrOutputParser()
)

In [ ]:
# complete RAG CHAIN - Re-initialized to pick up the updated response chain
rag_chain_w_sources = (
    {
        "context": similarity_retriever,
        "question": RunnablePassthrough()
    }
    | RunnablePassthrough.assign(
        response=src_rag_response_chain
    )
)

In [ ]:
# Re-test RAG with corrected model
query = "What is Artificial Intelligence?"
response = rag_chain_w_sources.invoke(query)
print(response.get('response'))

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 52.791343882s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
display_docs(response["context"])

Metadata : {'id': 10, 'title': 'Artificial Intelligence'}
Content :


Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. AI includes applications like virtual assistants, robotics, and autonomous vehicles. It's evolving rapidly with advancements in machine learning and deep learning.


Metadata : {'title': 'Natural Language Processing (NLP)', 'id': 3}
Content :


NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.


Metadata : {'id': 1, 'title': 'Machine Learning'}
Content :


Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.

In [ ]:
from deepeval.synthesizer import Synthesizer

In [ ]:
doc_contexts = []
for doc in processed_docs:
    doc_contexts.append(doc.page_content)
doc_contexts[:3]

['Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data. Algorithms analyze past data to make predictions or classify information. Popular applications include recommendation systems and image recognition.',
 'Deep learning is a subset of machine learning utilizing neural networks with many layers. It excels in complex tasks like image and speech recognition. Convolutional and recurrent neural networks are among the common architectures used.',
 'NLP is a branch of AI that enables computers to understand, interpret, and generate human language. Techniques include tokenization, stemming, and sentiment analysis. Applications range from chatbots to language translation services.']

In [ ]:
# Explicitly re-create Synthesizer with the corrected deepeval_gemini_model
synthesizer = Synthesizer(model=deepeval_gemini_model)

In [ ]:
# Re-attempting Golden Dataset generation with corrected model path
goldens = synthesizer.generate_goldens_from_contexts(
    contexts=[[doc] for doc in doc_contexts],
    include_expected_output=True,
    max_goldens_per_context=1
)
print(f"Successfully generated {len(goldens)} goldens.")

Output()

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 43.126331802s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.5-flash', 'location': 'global'},